# NURBS 3D solid — octant of a hollow sphere

Validates the NURBS (rational) path of the 3D solid kernels in `PatchIntegrator`,
added alongside the existing B-spline-only 3D solid work. A geometry rational in
only **one** parametric direction (e.g. a cylinder with a circular cross-section but
a straight extrusion) would under-test the 3D quotient rule: its weight function
$W(\xi)$ factors as $W_u(u) \cdot 1 \cdot 1$, so two of the three quotient-rule
derivatives ($\partial W/\partial v$, $\partial W/\partial w$) are identically zero
and never exercised.

**Geometry**: an octant of a **hollow sphere** (spherical shell wedge), built via
the standard NURBS "surface of revolution" technique (Piegl & Tiller, *The NURBS
Book*, §8.4): a quarter-circle **profile** curve (polar angle $u$, from the pole to
the equator) swept through a quarter-circle **rotation** (azimuthal angle $v$) about
the $z$-axis. The resulting weight is $w(u,v) = w_{profile}(u) \cdot w_{rot}(v)$ —
**both** angular directions carry the classic $w=1/\sqrt{2}$ corner weight, so
$W(u,v) = W_u(u) \cdot W_v(v)$ with **both** factors genuinely non-constant: now
$\partial W/\partial u$ depends on $v$ (through the $W_v(v)$ factor) and vice versa
— the two quotient-rule terms are properly *coupled*, which a single-direction
example cannot exercise. The radial (thickness) direction $w$ stays a plain affine
B-spline, the natural choice for an offset/thickness direction (no physical surface
needs weighting there) — mirroring the earlier cylinder notebook's extrusion
direction.

Three checks, none needing a closed-form displacement field (no legacy Fortran 3D
reference exists, same as the B-spline-only validation work).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (
    BSpline, BSplineVolume, Patch, ControlPointManager,
    GlobalDOFManager, PatchDOFManager, IGABasis1D, Material, Elasticity3D,
    PatchIntegrator, ConstantTraction3D,
)

## Geometry — surface of revolution

$P(i_u, i_v) = \big(x_p(i_u)\,R_x(i_v),\ x_p(i_u)\,R_y(i_v),\ z_p(i_u)\big)$,
$w(i_u,i_v) = w_p(i_u)\cdot w_{rot}(i_v)$ — the standard exact technique for
sweeping a profile curve through a circular arc. One copy of the profile at
$R=R_{in}$, one at $R=R_{ex}$; `local_shape = [3, 3, 2]` (u-fastest): 18 control
points.

In [ ]:
Rin, Rex = 1.0, 2.0
w0 = 1.0 / np.sqrt(2.0)
E, NU = 210000.0, 0.3
RHO = 7800.0

# Quarter-circle PROFILE (polar angle u: 0=pole on z-axis, 1=equator on x-axis).
def profile(R):
    return [(0.0, R, 1.0), (R, R, w0), (R, 0.0, 1.0)]   # (x, z, weight)

# Quarter-circle ROTATION (azimuthal angle v: 0=x-axis plane, 1=y-axis plane).
rot = [(1.0, 0.0, 1.0), (1.0, 1.0, w0), (0.0, 1.0, 1.0)]  # (Rx, Ry, weight)

mgr = ControlPointManager(dim=3)
n_cp = 3 * 3 * 2
for radius in (Rin, Rex):
    prof = profile(radius)
    for iv in range(3):
        rx, ry, wv = rot[iv]
        for iu in range(3):
            x_p, z_p, wu = prof[iu]
            mgr.add_point([x_p * rx, x_p * ry, z_p], w=wu * wv)

print(f'is_rational: {mgr.is_rational}, n_cp: {n_cp}')

su = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))   # polar angle, degree 2
sv = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))   # azimuthal angle, degree 2
sw = BSpline(1, np.array([0., 0., 1., 1.]))            # radial thickness, degree 1
tensor = BSplineVolume(su, sv, sw)

gdm = GlobalDOFManager([3] * n_cp)
pdm = PatchDOFManager(3, list(range(n_cp)), gdm)
patch = Patch(tensor, mgr, list(range(n_cp)), [3, 3, 2], pdm)

## Confirming the weight genuinely couples $u$ and $v$

Before trusting any downstream check, confirm directly (independent of
`PatchIntegrator`) that $W(u,v)$ is not separable into "rational in one direction,
trivial in the other": $\partial W/\partial u$ must depend on $v$.

In [ ]:
w_pattern = np.array([1.0, w0, 1.0])

def W_u(u):
    return w_pattern @ np.array([su.one_basis_fun(u, a) for a in range(3)])

def W_v(v):
    return w_pattern @ np.array([sv.one_basis_fun(v, a) for a in range(3)])

for u in [0.1, 0.3, 0.5, 0.7, 0.9]:
    print(f'W_u({u}) = {W_u(u):.6f}')

print()
print('W(u,v) = W_u(u) * W_v(v):')
for v in [0.1, 0.3]:
    print(f'  v={v}: W(0.1,v)={W_u(0.1)*W_v(v):.6f}  W(0.3,v)={W_u(0.3)*W_v(v):.6f}')
print('dW/du at fixed u changes with v (genuine coupling):',
      not np.isclose(W_v(0.1), W_v(0.3)))

## Visualization — control net vs. the true sphere

Evaluating the patch at many parametric points should trace a perfect sphere —
checking the evaluated radius against $R_{in}$/$R_{ex}$ is a strong, independent
correctness check of the rational geometry itself, before touching elasticity.

In [ ]:
def plot_patch_wireframe(patch, ax, color, linewidth=1.0, alpha=0.9, label=None):
    nu, nv, nw = patch.local_shape
    pts = np.array([patch.control_point(iu + iv*nu + iw*nu*nv)
                     for iw in range(nw) for iv in range(nv) for iu in range(nu)])
    pts = pts.reshape(nw, nv, nu, 3).transpose(2, 1, 0, 3)

    for iv in range(nv):
        for iw in range(nw):
            ax.plot(*pts[:, iv, iw, :].T, color=color, linewidth=linewidth, alpha=alpha)
    for iu in range(nu):
        for iw in range(nw):
            ax.plot(*pts[iu, :, iw, :].T, color=color, linewidth=linewidth, alpha=alpha)
    for iu in range(nu):
        for iv in range(nv):
            ax.plot(*pts[iu, iv, :, :].T, color=color, linewidth=linewidth, alpha=alpha)

    flat = pts.reshape(-1, 3)
    ax.scatter(*flat.T, color=color, s=10, alpha=alpha)
    if label:
        ax.text(*flat.mean(axis=0), label, color=color, fontsize=9, weight='bold')


fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection='3d')
plot_patch_wireframe(patch, ax, 'C0', label='control net')

n = 12
u_par = np.linspace(0.02, 0.98, n)
v_par = np.linspace(0.02, 0.98, n)
max_dev = {Rin: 0.0, Rex: 0.0}
for radius, w_par, c in [(Rin, 0.0, 'C1'), (Rex, 1.0, 'C3')]:
    pts = np.zeros((n, n, 3))
    for i, u in enumerate(u_par):
        for j, v in enumerate(v_par):
            span = [su.find_span(u), sv.find_span(v), sw.find_span(w_par)]
            pt = patch.evaluate_patch_nd_omp(
                np.array([span], dtype=np.int32), np.array([[u, v, w_par]]))[0]
            pts[i, j] = pt
            max_dev[radius] = max(max_dev[radius], abs(np.linalg.norm(pt) - radius))
    ax.plot_wireframe(pts[:, :, 0], pts[:, :, 1], pts[:, :, 2], color=c, linewidth=0.6, alpha=0.7)

ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_box_aspect([Rex, Rex, Rex])
ax.set_title('Octant of a hollow sphere: control net vs. true sphere')
plt.tight_layout()
plt.show()

for radius, dev in max_dev.items():
    print(f'R={radius}: max deviation of evaluated surface from a true sphere = {dev:.3e}')

## Stiffness — rigid-body patch test

Cheap, first correctness gate: the assembled stiffness matrix must be symmetric
and give exactly zero strain energy under any of the 6 rigid-body motions — now
genuinely exercising the coupled rational Jacobian in both angular directions.

In [ ]:
law = Elasticity3D(Material(E=E, nu=NU))
basis_u = IGABasis1D.build(su, su.degree + 1)
basis_v = IGABasis1D.build(sv, sv.degree + 1)
basis_w = IGABasis1D.build(sw, sw.degree + 1)
integrator = PatchIntegrator(patch, basis_u, basis_v, basis_w, law)

K = integrator.integrate_stiffness()
print(f'K: {K.shape}, nnz = {K.nnz}, symmetry error = {np.max(np.abs(K - K.T)):.2e}')

coords = np.array([patch.control_point(i) for i in range(n_cp)])
center = coords.mean(axis=0)

energies = []
for t in ([1., 0., 0.], [0., 1., 0.], [0., 0., 1.]):
    u = np.tile(t, n_cp)
    energies.append(u @ (K @ u))
for axis in ([1., 0., 0.], [0., 1., 0.], [0., 0., 1.]):
    u = np.zeros(3 * n_cp)
    for i in range(n_cp):
        u[3*i:3*i+3] = np.cross(axis, coords[i] - center)
    energies.append(u @ (K @ u))

print('Rigid-body mode energies (should all be ~0):')
print([f'{e:.2e}' for e in energies])

## Mass matrix — exact closed-form volume

$u^T M u = \rho \cdot \text{Volume}$ for a rigid unit-translation field, with
$\text{Volume} = \frac{\pi}{6}(R_{ex}^3 - R_{in}^3)$ (octant of a hollow sphere).

As with the earlier cylinder notebook, NURBS basis functions are rational, not
polynomial, so no finite Gauss rule reproduces this integral exactly — what's
checked is **convergence** as the Gauss point count increases.

In [ ]:
exact_volume = (np.pi / 6.0) * (Rex**3 - Rin**3)
exact_mass_x = RHO * exact_volume
u_rigid = np.tile([1.0, 0.0, 0.0], n_cp)

ngauss_list = [3, 4, 6, 8, 10, 14]
rel_errors = []
for ngauss in ngauss_list:
    bu = IGABasis1D.build(su, ngauss)
    bv = IGABasis1D.build(sv, ngauss)
    bw = IGABasis1D.build(sw, sw.degree + 1)
    integ = PatchIntegrator(patch, bu, bv, bw, Elasticity3D(Material(E=E, nu=NU, rho=RHO)))
    M = integ.integrate_mass()
    mass_x = u_rigid @ (M @ u_rigid)
    rel_err = abs(mass_x - exact_mass_x) / exact_mass_x
    rel_errors.append(rel_err)
    print(f'ngauss={ngauss:2d}: relative volume error = {rel_err:.3e}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(ngauss_list, rel_errors, 'o-')
ax.set_xlabel('Gauss points per direction (polar & azimuthal)')
ax.set_ylabel('relative volume error')
ax.set_title('Mass-matrix volume check: convergence with quadrature order')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Boundary load — resultant on a coupled face

The $v=0$ face (direction=1, side=0) lies in the **flat** $y=0$ plane (every
$v=0$ control point has $R_y=0$), but is still **curved within that plane** (a
quarter-circle arc swept by the polar angle $u$) — unlike the earlier cylinder
notebook's flat-and-non-rational face, this resultant is also a convergence check.
Area $= \frac{\pi}{4}(R_{ex}^2-R_{in}^2)$ (quarter-annulus cross-section).

In [ ]:
p = 1000.0
traction = ConstantTraction3D(np.array([0.0, 0.0, -p]))
exact_area = (np.pi / 4.0) * (Rex**2 - Rin**2)
exact_force = -p * exact_area

for ngauss in [3, 6, 10]:
    bu = IGABasis1D.build(su, ngauss)
    bv = IGABasis1D.build(sv, ngauss)
    bw = IGABasis1D.build(sw, sw.degree + 1)
    integ = PatchIntegrator(patch, bu, bv, bw, law)
    F = integ.integrate_boundary_load_3d(1, 0, traction)
    rel_err = abs(F[2::3].sum() - exact_force) / abs(exact_force)
    print(f'ngauss={ngauss:2d}: Fz = {F[2::3].sum():.6f}  '
          f'(exact {exact_force:.6f}, rel. error {rel_err:.3e})')
    print(f'           Fx, Fy (should be ~0) = {F[0::3].sum():.2e}, {F[1::3].sum():.2e}')